In [4]:
!pip install -q kagglehub tensorflow_hub transformers torch

import kagglehub
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold
import os
import time
from collections import defaultdict

In [5]:
path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")
print(f"Path to dataset files: {path}")

DATA_DIR = os.path.join(path, "Garbage classification", "Garbage classification")
class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
NUM_CLASSES = 6
IMG_SIZE = 224

print(f"\nChecking dataset at: {DATA_DIR}")
for class_name in class_names:
    class_dir = os.path.join(DATA_DIR, class_name)
    if os.path.exists(class_dir):
        num_imgs = len([f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  {class_name}: {num_imgs} images")
    else:
        print(f"  {class_name}: NOT FOUND")

Using Colab cache for faster access to the 'garbage-classification' dataset.
Path to dataset files: /kaggle/input/garbage-classification

Checking dataset at: /kaggle/input/garbage-classification/Garbage classification/Garbage classification
  cardboard: 403 images
  glass: 501 images
  metal: 410 images
  paper: 594 images
  plastic: 482 images
  trash: 137 images


In [6]:
from sklearn.model_selection import train_test_split
import pickle

file_paths = []
labels = []

for class_idx, class_name in enumerate(class_names):
    class_dir = os.path.join(DATA_DIR, class_name)
    for img_file in os.listdir(class_dir):
        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            file_paths.append(os.path.join(class_name, img_file))
            labels.append(class_idx)

print(f"Total images: {len(file_paths)}")

# 80/20 stratified split
train_paths, test_paths, train_labels, test_labels = train_test_split(
    file_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

print(f"\n Training pool (80%): {len(train_paths)} images")
print(f" Test pool (20%): {len(test_paths)} images - ISOLATED for later phases")

# Save test set paths for later
with open('test_set_isolated.pkl', 'wb') as f:
    pickle.dump({'paths': test_paths, 'labels': test_labels, 'class_names': class_names}, f)
print(" Test set saved to 'test_set_isolated.pkl' - DO NOT USE IN PHASE 1")

def print_dist(paths, labels, name):
    unique, counts = np.unique(labels, return_counts=True)
    print(f"\n{name} distribution:")
    for idx, count in zip(unique, counts):
        print(f"  {class_names[idx]}: {count} ({count/len(labels)*100:.1f}%)")

print_dist(train_paths, train_labels, "Training pool (for CV)")
print_dist(test_paths, test_labels, "Test pool (isolated)")

print("\n Variables defined: train_paths, train_labels, test_paths, test_labels")

Total images: 2527

 Training pool (80%): 2021 images
 Test pool (20%): 506 images - ISOLATED for later phases
 Test set saved to 'test_set_isolated.pkl' - DO NOT USE IN PHASE 1

Training pool (for CV) distribution:
  cardboard: 322 (15.9%)
  glass: 401 (19.8%)
  metal: 328 (16.2%)
  paper: 475 (23.5%)
  plastic: 385 (19.0%)
  trash: 110 (5.4%)

Test pool (isolated) distribution:
  cardboard: 81 (16.0%)
  glass: 100 (19.8%)
  metal: 82 (16.2%)
  paper: 119 (23.5%)
  plastic: 97 (19.2%)
  trash: 27 (5.3%)

 Variables defined: train_paths, train_labels, test_paths, test_labels


In [7]:
def preprocess_image(image_path, label):
    full_path = tf.strings.join([DATA_DIR, "/", image_path])
    image = tf.io.read_file(full_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def augment_cnn(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    return image, label

def create_cnn_dataset(paths, labels, batch_size, augment=False):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        dataset = dataset.map(augment_cnn, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    # Labels are integers (sparse) - convert to one-hot for CNNs
    dataset = dataset.map(lambda x, y: (x, tf.one_hot(y, depth=NUM_CLASSES)))
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [8]:
from tensorflow.keras import layers

def build_mobilenetv2(num_classes=6, learning_rate=1e-4, weight_decay=0.0001):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False

    model = tf.keras.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax',
                     kernel_regularizer=tf.keras.regularizers.l2(weight_decay))
    ])

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def build_resnet50v2(num_classes=6, learning_rate=0.001, weight_decay=0.0001):
    base = tf.keras.applications.ResNet50V2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False

    model = tf.keras.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax',
                     kernel_regularizer=tf.keras.regularizers.l2(weight_decay))
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate, weight_decay=weight_decay),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print(" CNN model builders ready")

 CNN model builders ready


In [9]:
from transformers import ViTModel
import torch

print("Loading pretrained weights from HuggingFace...")
pt_model = ViTModel.from_pretrained('google/vit-base-patch16-224')
pt_model.eval()
pt_weights = {name: param.detach().numpy() for name, param in pt_model.named_parameters()}
print(f" Loaded {len(pt_weights)} weight tensors")

def mlp_block(x, hidden_dim, out_dim, name):
    x = keras.layers.Dense(hidden_dim, activation='gelu', name=f"{name}_fc1")(x)
    x = keras.layers.Dense(out_dim, name=f"{name}_fc2")(x)
    return x

def attention_block(x, num_heads, head_dim, embed_dim, name):
    q = keras.layers.Dense(embed_dim, use_bias=True, name=f"{name}_q")(x)
    k = keras.layers.Dense(embed_dim, use_bias=True, name=f"{name}_k")(x)
    v = keras.layers.Dense(embed_dim, use_bias=True, name=f"{name}_v")(x)

    q = keras.layers.Reshape((-1, num_heads, head_dim), name=f"{name}_q_reshape")(q)
    k = keras.layers.Reshape((-1, num_heads, head_dim), name=f"{name}_k_reshape")(k)
    v = keras.layers.Reshape((-1, num_heads, head_dim), name=f"{name}_v_reshape")(v)

    q = keras.layers.Permute((2, 1, 3), name=f"{name}_q_perm")(q)
    k = keras.layers.Permute((2, 1, 3), name=f"{name}_k_perm")(k)
    v = keras.layers.Permute((2, 1, 3), name=f"{name}_v_perm")(v)

    scale = head_dim ** -0.5
    attn = keras.layers.Lambda(
        lambda qk: keras.ops.softmax(
            keras.ops.matmul(qk[0], keras.ops.transpose(qk[1], (0, 1, 3, 2))) * scale,
            axis=-1
        ),
        name=f"{name}_attn_scores"
    )([q, k])

    out = keras.layers.Lambda(
        lambda av: keras.ops.matmul(av[0], av[1]),
        name=f"{name}_attn_out"
    )([attn, v])

    out = keras.layers.Permute((2, 1, 3), name=f"{name}_out_perm")(out)
    out = keras.layers.Reshape((-1, embed_dim), name=f"{name}_out_reshape")(out)
    out = keras.layers.Dense(embed_dim, use_bias=True, name=f"{name}_proj")(out)
    return out

def transformer_block(x, num_heads=12, embed_dim=768, mlp_dim=3072, name="block"):
    y = keras.layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ln1")(x)
    y = attention_block(y, num_heads=num_heads, head_dim=embed_dim // num_heads,
                        embed_dim=embed_dim, name=f"{name}_attn")
    x = keras.layers.Add()([x, y])
    y = keras.layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ln2")(x)
    y = mlp_block(y, hidden_dim=mlp_dim, out_dim=embed_dim, name=f"{name}_mlp")
    x = keras.layers.Add()([x, y])
    return x

def build_vit_model(num_classes=6, learning_rate=2e-5, weight_decay=0.01):
    num_patches = (IMG_SIZE // 16) ** 2
    seq_len = num_patches + 1
    embed_dim = 768
    num_heads = 12
    mlp_dim = 3072
    num_layers = 12

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="pixel_values")

    patches = keras.layers.Conv2D(
        filters=embed_dim, kernel_size=16, strides=16,
        padding='valid', use_bias=True, name="patch_embed_proj"
    )(inputs)
    patches = keras.layers.Reshape((num_patches, embed_dim), name="patch_embed_reshape")(patches)

    class CLSToken(keras.layers.Layer):
        def build(self, input_shape):
            self.cls = self.add_weight(
                name="cls", shape=(1, 1, embed_dim),
                initializer="zeros", trainable=True
            )
        def call(self, x):
            return keras.ops.broadcast_to(self.cls, (keras.ops.shape(x)[0], 1, embed_dim))

    cls_tokens = CLSToken(name="cls_token")(patches)
    x = keras.layers.Concatenate(axis=1, name="cls_concat")([cls_tokens, patches])

    class PosEmbed(keras.layers.Layer):
        def build(self, input_shape):
            self.pos = self.add_weight(
                name="pos", shape=(1, seq_len, embed_dim),
                initializer="zeros", trainable=True
            )
        def call(self, x):
            return x + self.pos

    x = PosEmbed(name="pos_embed")(x)

    for i in range(num_layers):
        x = transformer_block(x, num_heads=num_heads, embed_dim=embed_dim,
                               mlp_dim=mlp_dim, name=f"block_{i}")

    x = keras.layers.LayerNormalization(epsilon=1e-6, name="ln_final")(x)
    cls_out = keras.layers.Lambda(lambda t: t[:, 0, :], name="cls_extract")(x)

    z = keras.layers.Dense(256, activation='relu', name="head_dense")(cls_out)
    z = keras.layers.Dropout(0.3, name="head_dropout")(z)
    outputs = keras.layers.Dense(num_classes, name="head_out")(z)

    vit_model = keras.Model(inputs=inputs, outputs=outputs)

    # Transfer pretrained weights
    print("Transferring pretrained weights...")

    def get_pt(key):
        return pt_weights[key]

    vit_model.get_layer("patch_embed_proj").set_weights([
        np.transpose(get_pt("embeddings.patch_embeddings.projection.weight"), (2, 3, 1, 0)),
        get_pt("embeddings.patch_embeddings.projection.bias")
    ])
    vit_model.get_layer("cls_token").set_weights([
        get_pt("embeddings.cls_token").reshape(1, 1, 768)
    ])
    vit_model.get_layer("pos_embed").set_weights([
        get_pt("embeddings.position_embeddings").reshape(1, 197, 768)
    ])

    for i in range(12):
        p = f"layers.{i}"
        vit_model.get_layer(f"block_{i}_ln1").set_weights([
            get_pt(f"{p}.layernorm_before.weight"),
            get_pt(f"{p}.layernorm_before.bias")
        ])
        vit_model.get_layer(f"block_{i}_attn_q").set_weights([
            get_pt(f"{p}.attention.q_proj.weight").T,
            get_pt(f"{p}.attention.q_proj.bias")
        ])
        vit_model.get_layer(f"block_{i}_attn_k").set_weights([
            get_pt(f"{p}.attention.k_proj.weight").T,
            get_pt(f"{p}.attention.k_proj.bias")
        ])
        vit_model.get_layer(f"block_{i}_attn_v").set_weights([
            get_pt(f"{p}.attention.v_proj.weight").T,
            get_pt(f"{p}.attention.v_proj.bias")
        ])
        vit_model.get_layer(f"block_{i}_attn_proj").set_weights([
            get_pt(f"{p}.attention.o_proj.weight").T,
            get_pt(f"{p}.attention.o_proj.bias")
        ])
        vit_model.get_layer(f"block_{i}_ln2").set_weights([
            get_pt(f"{p}.layernorm_after.weight"),
            get_pt(f"{p}.layernorm_after.bias")
        ])
        vit_model.get_layer(f"block_{i}_mlp_fc1").set_weights([
            get_pt(f"{p}.mlp.fc1.weight").T,
            get_pt(f"{p}.mlp.fc1.bias")
        ])
        vit_model.get_layer(f"block_{i}_mlp_fc2").set_weights([
            get_pt(f"{p}.mlp.fc2.weight").T,
            get_pt(f"{p}.mlp.fc2.bias")
        ])

    vit_model.get_layer("ln_final").set_weights([
        get_pt("layernorm.weight"),
        get_pt("layernorm.bias")
    ])
    print(" Pretrained weights transferred")

    # Freeze backbone
    backbone_layers = [l for l in vit_model.layers if not l.name.startswith("head_")]
    for layer in backbone_layers:
        layer.trainable = False

    vit_model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )

    trainable = sum([tf.size(v).numpy() for v in vit_model.trainable_variables])
    non_trainable = sum([tf.size(v).numpy() for v in vit_model.non_trainable_variables])
    print(f"Trainable:     {trainable:,}")
    print(f"Non-trainable: {non_trainable:,}")
    print(f"Total:         {trainable + non_trainable:,}")

    return vit_model

print(" ViT model builder ready")

Loading pretrained weights from HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 Loaded 200 weight tensors
 ViT model builder ready


In [10]:
def cross_validate_model(model_builder, model_name, train_paths, train_labels,
                          batch_size, is_vit=False, num_folds=5):

    skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

    fold_metrics = {
        'macro_f1': [], 'accuracy': [], 'precision': [], 'recall': [],
        'latency_ms': [], 'history': []
    }

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_paths, train_labels)):
        print(f"\n{'='*50}")
        print(f"Fold {fold+1}/{num_folds} - {model_name}")
        print(f"{'='*50}")

        # Split data
        fold_train_paths = [train_paths[i] for i in train_idx]
        fold_train_labels = [train_labels[i] for i in train_idx]
        fold_val_paths = [train_paths[i] for i in val_idx]
        fold_val_labels = [train_labels[i] for i in val_idx]

        print(f"Train: {len(fold_train_paths)}, Val: {len(fold_val_paths)}")

        # Create datasets
        if is_vit:
            # ViT uses sparse labels, smaller batch size
            train_ds = create_cnn_dataset(fold_train_paths, fold_train_labels, batch_size, augment=True)
            val_ds = create_cnn_dataset(fold_val_paths, fold_val_labels, batch_size, augment=False)
            # Convert to sparse labels for ViT
            train_ds = train_ds.map(lambda x, y: (x, tf.argmax(y, axis=1)))
            val_ds = val_ds.map(lambda x, y: (x, tf.argmax(y, axis=1)))
        else:
            train_ds = create_cnn_dataset(fold_train_paths, fold_train_labels, batch_size, augment=True)
            val_ds = create_cnn_dataset(fold_val_paths, fold_val_labels, batch_size, augment=False)

        model = model_builder()

        # Callbacks
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True, verbose=0
        )
        reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, verbose=0
        )

        # Measure inference latency (per image)
        def measure_latency(model, dataset, num_samples=100):
            latencies = []
            for images, _ in dataset.take(1):
                images = images[:min(num_samples, len(images))]
                for i in range(len(images)):
                    single_img = images[i:i+1]
                    start = time.perf_counter()
                    _ = model.predict(single_img, verbose=0)
                    latencies.append((time.perf_counter() - start) * 1000)  # ms
            return np.mean(latencies)

        history = model.fit(
            train_ds, validation_data=val_ds, epochs=20,
            callbacks=[early_stop, reduce_lr], verbose=1
        )

        y_true, y_pred = [], []
        for images, labels in val_ds:
            preds = model.predict(images, verbose=0)
            if is_vit:
                y_true.extend(labels.numpy())
                y_pred.extend(np.argmax(preds, axis=1))
            else:
                y_true.extend(np.argmax(labels.numpy(), axis=1))
                y_pred.extend(np.argmax(preds, axis=1))

        from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
        macro_f1 = f1_score(y_true, y_pred, average='macro')
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro')
        recall = recall_score(y_true, y_pred, average='macro')

        latency = measure_latency(model, val_ds)

        fold_metrics['macro_f1'].append(macro_f1)
        fold_metrics['accuracy'].append(accuracy)
        fold_metrics['precision'].append(precision)
        fold_metrics['recall'].append(recall)
        fold_metrics['latency_ms'].append(latency)
        fold_metrics['history'].append(history)

        print(f"Fold {fold+1} - Macro F1: {macro_f1:.4f}, Accuracy: {accuracy:.4f}, Latency: {latency:.2f}ms")

    print(f"\n{'='*50}")
    print(f"{model_name} - 5-Fold CV Results")
    print(f"{'='*50}")
    print(f"Macro F1:     {np.mean(fold_metrics['macro_f1']):.4f} ± {np.std(fold_metrics['macro_f1']):.4f}")
    print(f"Accuracy:     {np.mean(fold_metrics['accuracy']):.4f} ± {np.std(fold_metrics['accuracy']):.4f}")
    print(f"Precision:    {np.mean(fold_metrics['precision']):.4f} ± {np.std(fold_metrics['precision']):.4f}")
    print(f"Recall:       {np.mean(fold_metrics['recall']):.4f} ± {np.std(fold_metrics['recall']):.4f}")
    print(f"Latency:      {np.mean(fold_metrics['latency_ms']):.2f} ± {np.std(fold_metrics['latency_ms']):.2f} ms")

    return fold_metrics

In [10]:
batch_size_mobilenet = 32
batch_size_resnet = 32
batch_size_vit = 16

# MobileNetV2
print("\n" + "="*60)
print("STARTING 5-FOLD CV: MobileNetV2")
print("="*60)
mobilenet_results = cross_validate_model(
    lambda: build_mobilenetv2(NUM_CLASSES), "MobileNetV2",
    train_paths, train_labels, batch_size_mobilenet, is_vit=False
)

# ResNet50V2
print("\n" + "="*60)
print("STARTING 5-FOLD CV: ResNet50V2")
print("="*60)
resnet_results = cross_validate_model(
    lambda: build_resnet50v2(NUM_CLASSES), "ResNet50V2",
    train_paths, train_labels, batch_size_resnet, is_vit=False
)

# ViT
print("\n" + "="*60)
print("STARTING 5-FOLD CV: ViT")
print("="*60)
vit_results = cross_validate_model(
    lambda: build_vit_model(NUM_CLASSES), "ViT",
    train_paths, train_labels, batch_size_vit, is_vit=True
)


STARTING 5-FOLD CV: MobileNetV2

Fold 1/5 - MobileNetV2
Train: 1616, Val: 405
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 58s 661ms/step - accuracy: 0.1986 - loss: 2.1498 - val_accuracy: 0.2864 - val_loss: 1.7500 - learning_rate: 1.0000e-04
Epoch 2/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - accuracy: 0.3075 - loss: 1.7575 - val_accuracy: 0.4222 - val_loss: 1.5055 - learning_rate: 1.0000e-04
Epoch 3/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - accuracy: 0.3899 - loss: 1.5368 - val_accuracy: 0.4938 - val_loss: 1.3314 - learning_rate: 1.0000e-04
Epoch 4/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 4s 77ms/step - accuracy: 0.4876 - loss: 1.3742 - val_accuracy: 0.5580 - val_loss: 1.2035 - learning_rate: 1.0000e-04
Epoch 5/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - accuracy: 0.5093 - loss: 1.2752 - val_accuracy: 0.6025 - val_loss: 1.1052 - learning_rate: 1.0000e-04
Epoch 6/20
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 60ms/step - accuracy: 0.5668 - loss: 1.1556 - val_accu

ValueError: You must call `compile()` before using the model.

In [11]:
batch_size_vit = 16
# ViT
print("\n" + "="*60)
print("STARTING 5-FOLD CV: ViT")
print("="*60)
vit_results = cross_validate_model(
    lambda: build_vit_model(NUM_CLASSES), "ViT",
    train_paths, train_labels, batch_size_vit, is_vit=True
)


STARTING 5-FOLD CV: ViT

Fold 1/5 - ViT
Train: 1616, Val: 405
Transferring pretrained weights...
 Pretrained weights transferred
Trainable:     198,406
Non-trainable: 85,798,658
Total:         85,997,064
Epoch 1/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 63s 292ms/step - accuracy: 0.2766 - loss: 1.9554 - val_accuracy: 0.5432 - val_loss: 1.3153 - learning_rate: 2.0000e-05
Epoch 2/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 22s 219ms/step - accuracy: 0.4796 - loss: 1.3512 - val_accuracy: 0.6568 - val_loss: 1.0042 - learning_rate: 2.0000e-05
Epoch 3/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 42s 228ms/step - accuracy: 0.6052 - loss: 1.0687 - val_accuracy: 0.7210 - val_loss: 0.8539 - learning_rate: 2.0000e-05
Epoch 4/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 24s 237ms/step - accuracy: 0.6702 - loss: 0.9202 - val_accuracy: 0.7506 - val_loss: 0.7648 - learning_rate: 2.0000e-05
Epoch 5/20
101/101 ━━━━━━━━━━━━━━━━━━━━ 25s 247ms/step - accuracy: 0.7061 - loss: 0.8093 - val_accuracy: 0.7778 - val_loss: 0.7026 - learning_rate: 2.0000e-05


In [ ]:
import pandas as pd
import numpy as np

results_df = pd.DataFrame({
    'Model': ['MobileNetV2', 'ResNet50V2', 'ViT'],
    'Macro F1 (mean±std)': [
        f"{np.mean(mobilenet_results['macro_f1']):.4f} ± {np.std(mobilenet_results['macro_f1']):.4f}",
        f"{np.mean(resnet_results['macro_f1']):.4f} ± {np.std(resnet_results['macro_f1']):.4f}",
        f"{np.mean(vit_results['macro_f1']):.4f} ± {np.std(vit_results['macro_f1']):.4f}"
    ],
    'Accuracy': [
        f"{np.mean(mobilenet_results['accuracy']):.4f}",
        f"{np.mean(resnet_results['accuracy']):.4f}",
        f"{np.mean(vit_results['accuracy']):.4f}"
    ],
    'Latency (ms)': [
        f"{np.mean(mobilenet_results['latency_ms']):.2f}",
        f"{np.mean(resnet_results['latency_ms']):.2f}",
        f"{np.mean(vit_results['latency_ms']):.2f}"
    ]
})

print("\n" + "="*60)
print("PHASE 1 - FINAL RESULTS (5-Fold Cross-Validation)")
print("="*60)
print(results_df.to_string(index=False))

results_df.to_csv('phase1_cv_results.csv', index=False)
print("\n Results saved to 'phase1_cv_results.csv'")